# Submit the DNA full-cell simulation on the Delta Gateway

This notebook copies the workshop template into **your** bgvl folder and then submits the ~14 hour GPU simulation as a **Slurm batch job** that runs inside the `DNA_summer2025.sif` Apptainer image.

> **Why Slurm + Apptainer (and not a plain background job)?**
> The `btree_chromo` build shipped in the Gateway container is the newer **4DWCM** version, which no longer understands the loop-extrusion (`translocate`) commands this 2025 tutorial uses. The matching `btree_chromo` lives **only inside `DNA_summer2025.sif`**, and Apptainer can only run on a Delta compute/login node — not inside the Gateway's Jupyter container. So the job is launched with `sbatch` against the SIF.

- **Allocation:** `bgvl-delta-gpu`, partition `gpuA100x4`, **1 GPU**, ~15 h walltime
- **Kernel:** default Python (this notebook only sets up files and monitors)
- **Background reading:** [`README.md`](README.md) sections 6–12
- **VMD:** README section 12 (Open OnDemand Desktop)

**Workflow:**
1. Run sections 1–3 here on the Gateway to copy the template into your bgvl folder.
2. `sbatch` is **not** available inside the Gateway container, so submit the job from a Delta **SSH login** using the command section 4 prints for you.
3. Come back here and run section 5 to watch the log (your bgvl folder is shared, so the log is visible from the Gateway).

---
## 1. Setting paths

Set **`DELTA_USER`** in the cell below to your **NCSA username** (e.g. `alfiaparvez`) — this is your Delta SSH login and the name of your personal folder under `/projects/bgvl/`. This is **not** your Gateway/JupyterHub name (e.g. `alfiap-illinois`); on the Gateway everyone runs as a shared service account, so it cannot be detected automatically.

Each participant gets their own writable workspace on bgvl:

```
/projects/bgvl/<DELTA_USER>/DNA_SummerSchool_2026/
├── scripts/
├── data/                 ← simulation output (trajectory, coords, …)
└── DNA_tutorial.log      ← Slurm job log (created when the job starts)
```

Shared read-only workshop files (template source, Apptainer image, `launch_simulation.sh`) live under `/projects/bgvl/SummerSchool_2026/DNA/files/`.

In [ ]:
import os
import subprocess
from pathlib import Path

# Move to a directory that is guaranteed to exist (the kernel's cwd may have been
# deleted if you re-cloned the repo while this notebook was open).
for _safe in ("/home/user/workspace", "/projects/bgvl", "/tmp"):
    if os.path.isdir(_safe):
        os.chdir(_safe)
        break

GATEWAY_USER = os.environ.get("USER", "")

# --- Your Delta username -----------------------------------------------------
# Set this to your NCSA username (e.g. "alfiaparvez") — this is your Delta SSH
# login, the name of your personal folder under /projects/bgvl/, and the name
# you use for:
#     ssh <ncsa-username>@login.delta.ncsa.illinois.edu
# It is NOT your Gateway/JupyterHub name (e.g. "alfiap-illinois"): on the Gateway
# everyone runs as a shared service account, so it cannot be detected
# automatically. You must set it explicitly.
DELTA_USER = ""   # <-- your NCSA username, e.g. "alfiaparvez"

HOME = Path("/home/user/workspace")
REPO = HOME / "SummerSchool_2026" / "DNA"
BGVL_DNA = Path("/projects/bgvl/SummerSchool_2026/DNA")
PRELAUNCH = BGVL_DNA / "files" / "prelaunch_dna_workshop.sh"
LAUNCH = BGVL_DNA / "files" / "launch_simulation.sh"
SIF = BGVL_DNA / "files" / "DNA_summer2025.sif"

if not DELTA_USER:
    raise ValueError(
        "Set DELTA_USER above to your Delta SSH login name "
        f"(NOT your Gateway name '{GATEWAY_USER}'). "
        "It is the name of your personal folder under /projects/bgvl/."
    )

WORK_ROOT = Path(f"/projects/bgvl/{DELTA_USER}")
if not WORK_ROOT.is_dir():
    raise FileNotFoundError(
        f"{WORK_ROOT} does not exist — check DELTA_USER. Your personal folder "
        "under /projects/bgvl/ is named after your Delta login (run `ls /projects/bgvl`)."
    )
if not os.access(WORK_ROOT, os.W_OK):
    raise PermissionError(
        f"{WORK_ROOT} is not writable from this session — check DELTA_USER."
    )

SIM_ROOT = WORK_ROOT / "DNA_SummerSchool_2026"
SCRIPTS = SIM_ROOT / "scripts"
JOB_LOG = SIM_ROOT / "DNA_tutorial.log"

LOGIN_NODE = "login.delta.ncsa.illinois.edu"

if not REPO.is_dir() and not PRELAUNCH.is_file():
    subprocess.run(
        ["git", "clone", "https://github.com/Luthey-Schulten-Lab/SummerSchool_2026.git"],
        cwd=HOME,
        check=True,
    )

print("Delta username   :", DELTA_USER)
print("Gateway username :", GATEWAY_USER, "(not used for paths)")
print("Personal sim dir :", SIM_ROOT)
print("Slurm job log    :", JOB_LOG)

---
### Environment check (optional)

The only thing this notebook needs is the **Apptainer image** `DNA_summer2025.sif` on shared bgvl storage — the Slurm job runs `btree_chromo` *inside* it.

Note: the `/Software/btree_chromo` inside the Gateway container is the newer **4DWCM** build and is **not** used here (it can't run this tutorial's `translocate` commands).

In [ ]:
import sys

print("=" * 60)
print("Environment check")
print("=" * 60)
print(f"\nPython: {sys.executable}")
print(f"Version: {sys.version.split()[0]}")

print("\n--- Required for the Slurm job ---")
print(f"Apptainer SIF:        {'OK' if SIF.is_file() else 'NOT FOUND'}  ({SIF})")
print(f"launch_simulation.sh: {'OK' if LAUNCH.is_file() else 'NOT FOUND'}  ({LAUNCH})")

import shutil
print("\n--- Slurm client (in THIS Gateway session) ---")
print(f"sbatch:  {shutil.which('sbatch') or 'NOT FOUND (submit from a Delta SSH login — see section 4)'}")
print(f"squeue:  {shutil.which('squeue') or 'NOT FOUND'}")
print("\n" + "=" * 60)

---
## 2. Copy the workshop template to your bgvl directory

Run this cell **once in your own Gateway session**. It copies the read-only template from the shared workshop folder into **your** directory only:

```
/projects/bgvl/$USER/DNA_SummerSchool_2026/
```

Other participants have separate folders under `/projects/bgvl/<their-username>/` and do not share your simulation output.

In [ ]:
prelaunch = REPO / "files" / "prelaunch_dna_workshop.sh"
if not prelaunch.is_file():
    prelaunch = PRELAUNCH
if not prelaunch.is_file():
    raise FileNotFoundError(f"prelaunch script not found: {prelaunch}")

subprocess.run(["bash", str(prelaunch), DELTA_USER], check=True)
print(f"Copied template for user {DELTA_USER} → {WORK_ROOT}")
print("Scripts:", list(SCRIPTS.glob("*")))

---
## 3. Submit the simulation as a Slurm GPU job

The job runs `run_sc_chain_generation.sh` then `run_btree_chromo.py` **inside `DNA_summer2025.sif`** (91 biological minutes, ~14 h on an A100). It is defined in [`files/launch_simulation.sh`](files/launch_simulation.sh).

Run the cell below: if `sbatch` exists in this session it submits for you; otherwise it prints the exact commands to run from a **Delta SSH login** (the Gateway container has no Slurm client).

In [ ]:
import shutil

sbatch = shutil.which("sbatch")
# Pass the absolute workspace path + log path so the job does not depend on the
# username (Gateway name != Delta login name).
sbatch_cmd = [str(LAUNCH), str(SIM_ROOT)]

if sbatch:
    result = subprocess.run(
        [sbatch, f"--output={JOB_LOG}", *sbatch_cmd],
        capture_output=True, text=True,
    )
    print(result.stdout or "", result.stderr or "")
    if result.returncode == 0:
        print("Submitted. Track progress in section 5 (log) and section 4 (queue).")
    else:
        print("sbatch failed — submit from a Delta SSH login instead (see below).")
else:
    print("`sbatch` is not available in this Gateway session.")
    print("Submit the job from a Delta login node.")
    print("Use YOUR NCSA login name below (it may differ from your Gateway name):\n")
    print(f"    ssh <your-NCSA-username>@{LOGIN_NODE}")
    print(f"    sbatch --output={JOB_LOG} \\")
    print(f"           {LAUNCH} \\")
    print(f"           {SIM_ROOT}\n")
    print("Then return here and run section 5 to watch the log.")

---
## 4. Check the queue

Shows your job's state (`PD` pending, `R` running). Uses `squeue` if available here, otherwise prints the SSH command to check from a Delta login.

In [ ]:
squeue = shutil.which("squeue")
if squeue:
    subprocess.run([squeue, "--me", "-o", "%.18i %.12j %.8T %.10M %.6D %R"], check=False)
else:
    print("`squeue` not available here. Check from a Delta login:\n")
    print(f"    ssh <your-NCSA-username>@{LOGIN_NODE}")
    print(f"    squeue --me")

---
## 5. Track progress

Re-run the cells below while the simulation runs (~14 hours). The job log lives on shared bgvl storage, so it is visible from the Gateway even though the job runs on a compute node. The last lines show the current timestep.

In [ ]:
if JOB_LOG.is_file():
    subprocess.run(["tail", "-n", "80", str(JOB_LOG)], check=False)
else:
    print(f"No log yet (job may still be pending in the queue): {JOB_LOG}")

In [ ]:
traj = SIM_ROOT / "data" / "summerschool.lammpstrj"
print("Trajectory ready:", traj.is_file())
if traj.is_file():
    print(f"Size: {traj.stat().st_size / 1e6:.1f} MB")
print(traj)

---
## 6. Cancel the simulation (optional)

Use this if you submitted a job by mistake. Get the **job ID** from section 4 (`squeue`) and set it as `JOB_ID` below. If `scancel` is not available in this session, cancel from a Delta SSH login.

In [ ]:
if squeue:
    subprocess.run([squeue, "--me", "-o", "%.18i %.12j %.8T %.10M %R"], check=False)
else:
    print(f"Run on a Delta login:  ssh <your-NCSA-username>@{LOGIN_NODE}  then  squeue --me")

In [ ]:
JOB_ID = None  # e.g. 1234567 — set to the Slurm job ID from the cell above

if not JOB_ID:
    print("Set JOB_ID to your Slurm job ID, then re-run this cell.")
elif shutil.which("scancel"):
    r = subprocess.run(["scancel", str(JOB_ID)], capture_output=True, text=True)
    print(f"Cancelled job {JOB_ID}." if r.returncode == 0
          else f"scancel failed: {r.stderr or r.stdout}")
else:
    print(f"`scancel` not available here. Cancel from a Delta login:\n")
    print(f"    ssh <your-NCSA-username>@{LOGIN_NODE}")
    print(f"    scancel {JOB_ID}")